Importing libraries

In [1]:
import numpy as np
import os
import glob
import librosa
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# Download and extract the RAVDESS dataset
import urllib.request
import zipfile
import os

os.makedirs("RAVDESS", exist_ok=True)

# Using a public mirror of the RAVDESS Audio Speech dataset
url = "https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip?download=1"
zip_path = "RAVDESS.zip"

print("Downloading RAVDESS dataset (this may take a minute)... Woodlands...")
urllib.request.urlretrieve(url, zip_path)

print("Extracting dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("RAVDESS")

print("Done! Cleaning up zip file...")
os.remove(zip_path)
print("Dataset successfully loaded.")

Extracting dataset...
Done! Cleaning up zip file...
Dataset successfully loaded.


In [3]:
DATA_DIR = "RAVDESS/"

SR = 22050
N_MFCC = 40

emotions = {"01": "neutral", "02": "calm","03": "happy","04": "sad","05": "angry","06": "fearful","07": "disgust","08": "surprised"}

labels = list(emotions.values())

label_to_number = {"neutral": 0,"calm": 1,"happy": 2,"sad": 3,"angry": 4,"fearful": 5,"disgust": 6,"surprised": 7}

In [4]:
def extract_mfcc(file):

    audio, sr = librosa.load(file, sr=SR, duration=3)

    mfcc = librosa.feature.mfcc(y=audio,sr=sr, n_mfcc=N_MFCC)

    return mfcc.T

In [5]:
X = []
y = []

files = glob.glob(os.path.join(DATA_DIR, "**", "*.wav"),
    recursive=True
)

max_pad_len = 130

for file in files:

    filename = os.path.basename(file)

    parts = filename.replace(".wav", "").split("-")

    emotion_code = parts[2]

    mfcc = extract_mfcc(file)

    if mfcc.shape[0] < max_pad_len:
        pad_width = max_pad_len - mfcc.shape[0]
        mfcc = np.pad(mfcc, pad_width=((0, pad_width), (0, 0)), mode='constant')
    else:
        mfcc = mfcc[:max_pad_len, :]

    X.append(mfcc)

    y.append(label_to_number[emotions[emotion_code]])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1440, 130, 40)
y shape: (1440,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (1152, 130, 40)
Testing data: (288, 130, 40)


X = audio features

y = emotion/target

X_train = data used to train the model

X_test = unseen data used to test the model

In [7]:
model = Sequential()

model.add(Conv1D(64,5,activation="relu",input_shape=X_train.shape[1:]))

model.add(MaxPooling1D(2))
model.add(Conv1D(128, 5, activation="relu"))
model.add(MaxPooling1D(2))
model.add(Flatten())
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(8, activation="softmax"))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 126, 64)        │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 63, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 59, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 29, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3712)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       237,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 292,104 (1.11 MB)

 Trainable params: 292,104 (1.11 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = model.fit(X_train,y_train,epochs=30,batch_size=32,validation_split=0.2)

Epoch 1/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 60ms/step - accuracy: 0.1433 - loss: 24.7485 - val_accuracy: 0.1385 - val_loss: 2.0717
Epoch 2/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.1640 - loss: 2.0626 - val_accuracy: 0.1255 - val_loss: 2.0579
Epoch 3/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.1802 - loss: 2.0230 - val_accuracy: 0.1645 - val_loss: 2.2298
Epoch 4/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.1716 - loss: 2.0612 - val_accuracy: 0.1299 - val_loss: 2.0447
Epoch 5/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.1813 - loss: 2.0348 - val_accuracy: 0.1299 - val_loss: 2.0549
Epoch 6/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.1607 - loss: 2.0437 - val_accuracy: 0.1558 - val_loss: 2.0500
Epoch 7/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.1802 - loss: 2.0012 - val_accuracy: 0.1342 - val_loss: 2.0358
Epoch 8/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step - accuracy: 0.1846 - loss: 1.9942 - val_accuracy: 0.1429 - 

In [11]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.2118 - loss: 1.9145
Test Accuracy: 0.2118055522441864


Prediction

In [12]:
predictions = model.predict(X_test)

predicted_labels = np.argmax(predictions, axis=1)

print(predicted_labels)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
[1 1 1 1 5 5 4 4 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 5
 1 1 1 1 1 1 4 4 1 1 1 1 1 1 1 1 5 1 1 1 1 1 1 1 1 1 1 4 1 1 1 1 1 1 1 1 4
 1 1 5 1 1 4 1 1 1 1 1 1 1 1 1 1 1 1 5 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 4 1 1 5 1 1 1 5 1 1 1 5 1 1 1 1 1 5 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 5 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 5 1 1 4 1 1 1 1 1 1 1
 4 1 1 1 4 1 1 1 1 1 1 1 1 1 1 1 1 5 1 1 1 1 1 1 1 1 1 1 1 1 4 1 1 1 1 1 1
 1 4 4 1 4 4 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 5 1 1 1 1 4 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 5 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


Classification report

In [13]:
print(classification_report(y_test,predicted_labels,target_names=labels))

              precision    recall  f1-score   support

     neutral       0.00      0.00      0.00        19
        calm       0.15      1.00      0.26        38
       happy       0.00      0.00      0.00        38
         sad       0.00      0.00      0.00        38
       angry       0.94      0.41      0.57        39
     fearful       0.47      0.18      0.26        39
     disgust       0.00      0.00      0.00        38
   surprised       0.00      0.00      0.00        39

    accuracy                           0.21       288
   macro avg       0.19      0.20      0.14       288
weighted avg       0.21      0.21      0.15       288



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
cm = confusion_matrix(y_test,predicted_labels)
print(cm)

[[ 0 19  0  0  0  0  0  0]
 [ 0 38  0  0  0  0  0  0]
 [ 0 36  0  0  0  2  0  0]
 [ 0 38  0  0  0  0  0  0]
 [ 0 18  0  0 16  5  0  0]
 [ 0 31  0  0  1  7  0  0]
 [ 0 37  0  0  0  1  0  0]
 [ 0 39  0  0  0  0  0  0]]


In [15]:
print(np.unique(y))

[0 1 2 3 4 5 6 7]
